In [1]:
!pip install -U torch torchvision torchaudio
!pip install -U transformers accelerate peft bitsandbytes trl datasets qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.5/915.5 MB 60.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 46.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 56.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 66.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 76.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 104.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 96.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 109.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 111.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288

In [8]:
import torch
import gc

# Delete the model and processor variables if they exist
try:
    del model
    del processor
except NameError:
    pass

# Force Python's garbage collector to run
gc.collect()

# Empty the PyTorch CUDA cache
torch.cuda.empty_cache()

In [1]:
import os

# 1. Route all Hugging Face downloads to your spacious persistent volume
os.environ["HF_HOME"] = "/workspace/huggingface_cache"

# 2. Keep the fix for the Xet network timeout issue
os.environ["HF_HUB_DISABLE_XET"] = "1"

print(f"Hugging Face cache set to: {os.environ['HF_HOME']}")

Hugging Face cache set to: /workspace/huggingface_cache


In [3]:
!pip install thefuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.3 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
import json
import numpy as np
import json_repair # Assuming this is the library you are using for the repair logic
try:
    from thefuzz import fuzz
except ImportError:
    from fuzzywuzzy import fuzz

def _parse_mixed_value(val):
    if isinstance(val, (int, float)) and not isinstance(val, bool):
        return float(val), True
    if isinstance(val, str):
        clean_v = val.replace('−', '-').replace(',', '').strip()
        try:
            return float(clean_v), True
        except ValueError:
            return str(val).strip(), False
    return str(val), False

def _clean_math_equation(eq_str):
    if not isinstance(eq_str, str): return ""
    return eq_str.split('=', 1)[-1].strip().replace(" ", "")

def evaluate_json_accuracy(reference_json, model_json):
    # Initialize optional metrics to None. They will only become floats if evaluated.
    results = {
        "valid_json": False, 
        "schema_score": 0.0, 
        "topology_score": 0.0,
        "domain_accuracy": None, 
        "numerical_accuracy": None, 
        "categorical_accuracy": None,
        "math_accuracy": None, 
        "semantic_score": None
    }
    
    try:
        # Load the ground truth normally
        ref = json.loads(reference_json) if isinstance(reference_json, str) else reference_json
        
        # 🛡️ FOOLPROOF REPAIR LOGIC
        if isinstance(model_json, str):
            # return_objects=True forces it to return a Python dictionary, not a string
            pred = json_repair.repair_json(model_json, return_objects=True)
        else:
            pred = model_json
            
        if isinstance(pred, dict) and len(pred) > 0:
            results["valid_json"] = True
        else:
            # If repair failed to yield a dictionary, we skip evaluation for this chart
            return results
    except Exception as e:
        # 🚨 This will tell us EXACTLY what is crashing the script
        print(f"CRITICAL ERROR loading JSON: {str(e)}")
        return results

    # 1. SCHEMA & TRUNCATION
    ref_keys, pred_keys = set(ref.keys()), set(pred.keys())
    if ref_keys:
        results["schema_score"] = len(ref_keys & pred_keys) / len(ref_keys)
        
    if ref.get("is_truncated_for_context"):
        results["schema_score"] += 0.1 if pred.get("is_truncated_for_context") else -0.1
        results["schema_score"] = min(1.0, max(0.0, results["schema_score"]))

    # 2. TOPOLOGY
    rt = ref.get("topo", {})
    pt = pred.get("topo", {})
    # Fixed: pt.get("type") instead of pred.get("type")
    results["topology_score"] = (1.0 if rt.get("type") == pt.get("type") else 0.0)

    # 3. DOMAINS (Skipped for Pie charts unless hallucinated)
    ref_axes, pred_axes = ref.get("axes", []), pred.get("axes", [])
    if not ref_axes:
        results["domain_accuracy"] = 0.0 if pred_axes else None
    else:
        dom_scores = []
        for r_ax in ref_axes:
            p_ax = next((a for a in pred_axes if a.get("name") == r_ax.get("name")), {})
            for r_val, p_val in zip(r_ax.get("dom", []), p_ax.get("dom", [])):
                rv, r_num = _parse_mixed_value(r_val)
                pv, p_num = _parse_mixed_value(p_val)
                if r_num and p_num:
                    dom_scores.append(1.0 if abs(rv - pv) <= max(abs(rv) * 0.05, 1e-5) else 0.0)
                elif not r_num and not p_num:
                    dom_scores.append(1.0 if str(rv).lower() == str(pv).lower() else 0.0)
                else:
                    dom_scores.append(0.0)
                    
        if dom_scores:
            length_penalty = min(1.0, len(pred_axes) / max(1, len(ref_axes)))
            results["domain_accuracy"] = np.mean(dom_scores) * length_penalty

    # 4. MATH (Locked entirely to Line Charts per user constraint)
    if rt.get("type") == "line":
        r_eqs = [_clean_math_equation(eq) for eq in ref.get("math", [])]
        p_eqs = [_clean_math_equation(eq) for eq in pred.get("math", [])]
        if r_eqs:
            results["math_accuracy"] = np.mean([max([fuzz.ratio(r, p) for p in p_eqs] + [0]) / 100.0 for r in r_eqs])
        elif p_eqs:
            results["math_accuracy"] = 0.0  # Penalize hallucination
    else:
        # Strictly ignored for non-Line charts regardless of extraction
        results["math_accuracy"] = None 

    # 5. SERIES DATA (Numerical vs Categorical Separation)
    num_errors, cat_matches = [], []
    ref_ser, pred_ser = ref.get("ser", []), pred.get("ser", [])
    
    for r_s in ref_ser:
        p_s = next((s for s in pred_ser if s.get("name") == r_s.get("name")), {})
        for r_pt, p_pt in zip(r_s.get("data", []), p_s.get("data", [])):
            if not isinstance(r_pt, (list, tuple)) or not isinstance(p_pt, (list, tuple)): continue
            
            for rv_raw, pv_raw in zip(r_pt, p_pt):
                rv, r_num = _parse_mixed_value(rv_raw)
                pv, p_num = _parse_mixed_value(pv_raw)
                
                if r_num and p_num:
                    denominator = max(abs(rv), abs(pv), 1e-5)
                    num_errors.append(min(1.0, abs(rv - pv) / denominator))
                elif not r_num and not p_num:
                    cat_matches.append(1.0 if str(rv).lower() == str(pv).lower() else 0.0)
                else:
                    if r_num:
                        num_errors.append(1.0) 
                    else:
                        cat_matches.append(0.0)

    r_total_pts = sum(len(s.get("data", [])) for s in ref_ser)
    p_total_pts = sum(len(s.get("data", [])) for s in pred_ser)
    
    if r_total_pts > 0 and num_errors:
        length_penalty = min(r_total_pts, p_total_pts) / max(1, max(r_total_pts, p_total_pts))
        results["numerical_accuracy"] = max(0.0, (1.0 - np.mean(num_errors)) * length_penalty)

    # Categorical accuracy will safely remain None for Histograms
    if cat_matches:
        results["categorical_accuracy"] = np.mean(cat_matches)

    # 6. EXTENDED SEMANTICS (Skipped if chart has no trends/stats to evaluate)
    sem_checks = []
    for r_s in ref_ser:
        p_s = next((s for s in pred_ser if s.get("name") == r_s.get("name")), {})
        
        r_trend = r_s.get("trend", {}).get("global_trend")
        if r_trend:
            sem_checks.append(1.0 if r_trend == p_s.get("trend", {}).get("global_trend") else 0.0)
            
        # Added Area chart bounds to the stat_key check list
        for stat_key in ["max_value", "min_value", "max_frequency", "max_upper_bound", "min_upper_bound"]:
            r_stat_val = r_s.get("stats", {}).get(stat_key)
            if r_stat_val is not None:
                rv, r_num = _parse_mixed_value(r_stat_val)
                pv, p_num = _parse_mixed_value(p_s.get("stats", {}).get(stat_key))
                if r_num and p_num:
                    sem_checks.append(1.0 if abs(rv - pv) <= max(abs(rv) * 0.05, 1e-5) else 0.0)
                else:
                    sem_checks.append(0.0)

        r_ds, p_ds = r_s.get("distribution_summary", {}), p_s.get("distribution_summary", {})
        if "original_bin_count" in r_ds:
            sem_checks.append(1.0 if r_ds["original_bin_count"] == p_ds.get("original_bin_count") else 0.0)
            
        r_cs, p_cs = r_s.get("cluster_statistics", {}), p_s.get("cluster_statistics", {})
        if "total_points" in r_cs:
            sem_checks.append(1.0 if r_cs["total_points"] == p_cs.get("total_points") else 0.0)

    ref_rels = " ".join([r.get("relation", r.get("description", "")) for r in ref.get("rel", [])])
    pred_rels = " ".join([r.get("relation", r.get("description", "")) for r in pred.get("rel", [])])
    if ref_rels:
        sem_checks.append(fuzz.ratio(ref_rels, pred_rels) / 100.0)

    if sem_checks:
        results["semantic_score"] = np.mean(sem_checks)

    return results

def extract_numbers(d):
    nums = []
    if isinstance(d, dict):
        for v in d.values(): nums.extend(extract_numbers(v))
    elif isinstance(d, list):
        for v in d: nums.extend(extract_numbers(v))
    elif isinstance(d, (int, float)) and not isinstance(d, bool):
        nums.append(float(d))
    return nums

In [ ]:
import json
import os
import torch
from peft import PeftModel
from transformers import AutoProcessor, AutoModelForImageTextToText
from qwen_vl_utils import process_vision_info

# --- Configuration Paths ---
BASE_MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
ADAPTER_PATH = "/workspace/qwen_lora_final/qwen_lora_final"
TEST_JSON_PATH = "extracted_chart_specs_test.json"
TEST_IMG_DIR = "./test_images_300"
PREDICTIONS_OUTPUT = "predicted_specs_test.json"

TARGET_CHART_TYPES = []

# ==========================================
# 1. HIGHLY SPECIFIC SCHEMAS
# ==========================================
# ==========================================
# POLYMORPHIC HYBRID SCHEMAS (UPDATED FOR EDGE CASES)
# ==========================================

SCHEMA_PROMPT_BAR = """Extract the data, topology, and relational structure of this chart into a JSON object matching this exact schema. Omit any keys that have empty lists, empty objects, or null values.

{
  "title": "Chart title or null",
  "topo": {"type": "bar", "sub": "vertical/horizontal/standard"},
  "legend": {"on": true/false, "title": "Legend title or null"},
  "axes": [{"name": "x_axis / y_axis / y_axis_1", "lbl": "Axis label or null", "scl": "linear/log/categorical", "dom": ["min_val", "max_val"]}],
  "ser": [
    {
      "name": "Bar Series", "color": "#hexcode", "y_ref": "y_axis or y_axis_1",
      "data": [["x_category", "y_value"]],
      "trend": {"global_trend": "String", "rate_of_change": "String", "intervals": []},
      "stats": {"max_value": 0.0, "max_at_the_ref_point": "x_val", "min_value": 0.0, "min_at_the_ref_point": "x_val"}
    },
    {
      "name": "threshold_y_100", "color": "#hexcode", "y_ref": "y_axis or y_axis_1",
      "data": [["x_start", 100.0], ["x_end", 100.0]]
    },
    {
      "name": "scatter_series_optional", "color": "#hexcode", "y_ref": "y_axis or y_axis_1",
      "data": [["x_val", "y_val"]],
      "cluster_statistics": {"total_points": 0, "centroid": [0.0, 0.0], "spread_std_dev": [0.0, 0.0], "bounding_box": {"x_min": 0.0, "x_max": 0.0, "y_min": 0.0, "y_max": 0.0}, "outliers": []}
    }
  ],
  "rel": [{"relation_type": "String", "description": "String", "ranking": ["Series A", "Series B"]}],
  "is_truncated_for_context": true/false,
  "sampling_strategy": "String or null",
  "total_original_series": 0,
  "note": "String or null"
}

Format Rules:
- SECONDARY AXES: For dual-axis charts, map secondary axes to 'y_axis_1' and use it in 'y_ref'.
- HYBRID/THRESHOLDS: If the chart contains a horizontal or vertical threshold line, extract it as a separate series with exactly 2 data points representing its start and end.
- HYBRID CHARTS: If the chart contains scatter points or outliers, represent them as a separate series with 'cluster_statistics'.
- Return ONLY the raw JSON object. Do not use markdown formatting blocks (```)."""

SCHEMA_PROMPT_AREA = """Extract the data, topology, and relational structure of this chart into a JSON object matching this exact schema. Omit any keys that have empty lists, empty objects, or null values.

{
  "title": "Chart title or null",
  "topo": {"type": "area", "sub": "stacked/unstacked"},
  "legend": {"on": true/false, "title": "Legend title or null"},
  "math": ["Generating expressions extracted from code (e.g., 'y = np.sin(x)')"],
  "axes": [{"name": "x_axis / y_axis / y_axis_1", "lbl": "Axis label or null", "scl": "linear/log/categorical", "dom": ["min_val", "max_val"]}],
  "ser": [
    {
      "name": "Area Series", "color": "#hexcode", "y_ref": "y_axis or y_axis_1",
      "is_math": true/false, "ds": true/false, "critical_points_retained": true/false,
      "data": [["x_val", "y_max", "y_min"]],
      "trend": {"global_trend": "String", "rate_of_change": "String", "intervals": []},
      "stats": {"max_upper_bound": 0.0, "max_at_the_ref_point": "x_val", "min_upper_bound": 0.0, "min_at_the_ref_point": "x_val", "crossing_points": []}
    },
    {
      "name": "scatter_series_optional", "color": "#hexcode", "y_ref": "y_axis or y_axis_1",
      "data": [["x_val", "y_val"]],
      "cluster_statistics": {"total_points": 0, "centroid": [0.0, 0.0], "spread_std_dev": [0.0, 0.0], "bounding_box": {"x_min": 0.0, "x_max": 0.0, "y_min": 0.0, "y_max": 0.0}, "outliers": []}
    }
  ],
  "rel": [{"relation_type": "String", "description": "String", "ranking": ["Series A", "Series B"]}],
  "is_truncated_for_context": true/false,
  "sampling_strategy": "String or null",
  "total_original_series": 0,
  "note": "String or null"
}

Format Rules:
- SECONDARY AXES: For dual-axis charts, map secondary axes to 'y_axis_1' and use it in 'y_ref'.
- MATH GENERATION: Include 'math' (top-level) and 'is_math' (series-level) if the area was generated by a detectable mathematical function.
- For Area bands, 'data' must be a 3-element list: ["x_val", "y_max", "y_min"].
- Return ONLY the raw JSON object. Do not use markdown formatting blocks (```)."""

SCHEMA_PROMPT_PIE = """Extract the data, topology, and relational structure of this chart into a JSON object matching this exact schema. Omit any keys that have empty lists, empty objects, or null values.

{
  "title": "Chart title or null",
  "topo": {"type": "pie", "sub": "standard/donut"},
  "legend": {"on": true/false, "title": "Legend title or null"},
  "axes": [],
  "ser": [
    {
      "name": "Slice Name", "color": "#hexcode or null", "y_ref": null,
      "data": [[0.0]]
    }
  ],
  "is_truncated_for_context": true/false,
  "sampling_strategy": "String or null",
  "total_original_series": 0
}

Format Rules:
- For Pie charts, 'data' represents the slice value or percentage inside a nested list.
- Return ONLY the raw JSON object. Do not use markdown formatting blocks (```)."""

SCHEMA_PROMPT_LINE = """Extract the data, topology, and relational structure of this chart into a JSON object matching this exact schema. Omit any keys that have empty lists, empty objects, or null values.

{
  "title": "Chart title or null",
  "topo": {"type": "line", "sub": "standard/hybrid scattered"},
  "legend": {"on": true/false, "title": "Legend title or null"},
  "math": ["Generating expressions extracted from code (e.g., 'y = np.sin(x)')"],
  "axes": [{"name": "x_axis / y_axis / y_axis_1", "lbl": "Axis label or null", "scl": "linear/log/categorical", "dom": ["min_val", "max_val"]}],
  "ser": [
    {
      "name": "Line Series", "color": "#hexcode", "y_ref": "y_axis or y_axis_1",
      "is_math": true/false, "ds": true/false,
      "data": [["x_val", "y_val"]],
      "trend": {"global_trend": "String", "rate_of_change": "String", "intervals": []},
      "stats": {"max_value": 0.0, "max_at_the_ref_point": "x_val", "min_value": 0.0, "min_at_the_ref_point": "x_val", "crossing_points": []}
    },
    {
      "name": "scatter_series_optional", "color": "#hexcode", "y_ref": "y_axis or y_axis_1",
      "data": [["x_val", "y_val"]],
      "cluster_statistics": {"total_points": 0, "centroid": [0.0, 0.0], "spread_std_dev": [0.0, 0.0], "bounding_box": {"x_min": 0.0, "x_max": 0.0, "y_min": 0.0, "y_max": 0.0}, "outliers": []}
    }
  ],
  "rel": [{"relation_type": "String", "description": "String", "ranking": ["Series A", "Series B"]}],
  "is_truncated_for_context": true/false,
  "sampling_strategy": "String or null",
  "total_original_series": 0,
  "note": "String or null"
}

Format Rules:
- SECONDARY AXES: For dual-axis charts, map secondary axes to 'y_axis_1' and use it in 'y_ref'.
- HYBRID CHARTS: If the chart contains scatter points alongside lines, represent them as a separate series with optional 'cluster_statistics'.
- Return ONLY the raw JSON object. Do not use markdown formatting blocks (```)."""

SCHEMA_PROMPT_HISTOGRAM = """Extract the data, topology, and relational structure of this chart into a JSON object matching this exact schema. Omit any keys that have empty lists, empty objects, or null values.

{
  "title": "Chart title or null",
  "topo": {"type": "histogram", "sub": "standard/density"},
  "legend": {"on": true/false, "title": "Legend title or null"},
  "axes": [{"name": "x_axis / y_axis / y_axis_1", "lbl": "Axis label or null", "scl": "linear/log/categorical", "dom": ["min_val", "max_val"]}],
  "ser": [
    {
      "name": "Histogram Series", "color": "#hexcode", "y_ref": "y_axis or y_axis_1",
      "ds": true/false,
      "data": [["bin_start", "bin_end", "height"]],
      "distribution_summary": {"original_bin_count": 0, "x_range": [0.0, 0.0], "max_frequency_recorded": 0.0},
      "stats": {"modal_bin": [0.0, 0.0], "max_frequency": 0.0}
    },
    {
      "name": "threshold_x_25", "color": "#hexcode", "y_ref": "y_axis or y_axis_1",
      "data": [[25.0, "y_min"], [25.0, "y_max"]]
    }
  ],
  "is_truncated_for_context": true/false,
  "sampling_strategy": "String or null",
  "total_original_series": 0,
  "note": "String or null"
}

Format Rules:
- SECONDARY AXES: For dual-axis charts, map secondary axes to 'y_axis_1' and use it in 'y_ref'.
- HYBRID/THRESHOLDS: If the chart contains a horizontal or vertical threshold line, extract it as a separate series with exactly 2 data points representing its start and end.
- Return ONLY the raw JSON object. Do not use markdown formatting blocks (```)."""

SCHEMA_PROMPT_BOX = """Extract the data, topology, and relational structure of this chart into a JSON object matching this exact schema. Omit any keys that have empty lists, empty objects, or null values.

{
  "title": "Chart title or null",
  "topo": {"type": "box", "sub": "vertical/horizontal"},
  "legend": {"on": true/false, "title": "Legend title or null"},
  "axes": [{"name": "x_axis / y_axis / y_axis_1", "lbl": "Axis label or null", "scl": "linear/log/categorical", "dom": ["min_val", "max_val"]}],
  "ser": [
    {
      "name": "Box Distribution", "color": "#hexcode", "y_ref": "y_axis or y_axis_1",
      "data": [["x_val", "min", "q1", "median", "q3", "max"]],
      "distribution_summary": {"average_iqr": 0.0, "box_details": [{"x_category": "val", "iqr": 0.0, "skewness": "symmetric/positive_skew/negative_skew"}]},
      "stats": {"max_value": 0.0, "min_value": 0.0}
    },
    {
      "name": "threshold_y_target", "color": "#hexcode", "y_ref": "y_axis or y_axis_1",
      "data": [["x_start", 50.0], ["x_end", 50.0]]
    },
    {
      "name": "scatter_series_outliers", "color": "#hexcode", "y_ref": "y_axis or y_axis_1",
      "data": [["x_val", "y_val"]],
      "cluster_statistics": {"total_points": 0, "centroid": [0.0, 0.0], "spread_std_dev": [0.0, 0.0], "bounding_box": {"x_min": 0.0, "x_max": 0.0, "y_min": 0.0, "y_max": 0.0}, "outliers": []}
    }
  ],
  "rel": [{"relation_type": "String", "description": "String", "ranking": ["Series A", "Series B"]}],
  "is_truncated_for_context": true/false,
  "sampling_strategy": "String or null",
  "total_original_series": 0,
  "note": "String or null"
}

Format Rules:
- SECONDARY AXES: For dual-axis charts, map secondary axes to 'y_axis_1' and use it in 'y_ref'.
- HYBRID/THRESHOLDS: If the chart contains a horizontal or vertical threshold line, extract it as a separate series with exactly 2 data points representing its start and end.
- HYBRID/OUTLIERS: Outliers (fliers) are explicitly extracted as a separate series named "scatter_series_outliers" using 2-element arrays [x, y].
- Return ONLY the raw JSON object. Do not use markdown formatting blocks (```)."""

SCHEMA_PROMPT_SCATTER = """Extract the data, topology, and relational structure of this chart into a JSON object matching this exact schema. Omit any keys that have empty lists, empty objects, or null values.

{
  "title": "Chart title or null",
  "topo": {"type": "scatter", "sub": "standard/bubble"},
  "legend": {"on": true/false, "title": "Legend title or null"},
  "math": ["Generating expressions extracted from code (e.g., 'y = np.sin(x)')"],
  "axes": [{"name": "x_axis / y_axis / y_axis_1", "lbl": "Axis label or null", "scl": "linear/log/categorical", "dom": ["min_val", "max_val"]}],
  "ser": [
    {
      "name": "Scatter Series", "color": "#hexcode", "y_ref": "y_axis or y_axis_1",
      "is_math": true/false, "is_subsampled": true/false,
      "data": [["x_val", "y_val"]],
      "cluster_statistics": {"total_points": 0, "centroid": [0.0, 0.0], "spread_std_dev": [0.0, 0.0], "bounding_box": {"x_min": 0.0, "x_max": 0.0, "y_min": 0.0, "y_max": 0.0}, "outliers": [["x", "y"]]}
    },
    {
      "name": "threshold_y_100", "color": "#hexcode", "y_ref": "y_axis or y_axis_1",
      "data": [["x_start", 100.0], ["x_end", 100.0]]
    },
    {
      "name": "Area_Bands_Optional", "color": "#hexcode", "y_ref": "y_axis or y_axis_1",
      "data": [["x_val", "y_max", "y_min"]],
      "stats": {"max_upper_bound": 0.0, "min_upper_bound": 0.0}
    }
  ],
  "rel": [{"relation_type": "String", "description": "String", "ranking": ["Series A", "Series B"]}],
  "is_truncated_for_context": true/false,
  "sampling_strategy": "String or null",
  "total_original_series": 0,
  "note": "String or null"
}

Format Rules:
- SECONDARY AXES: For dual-axis charts, map secondary axes to 'y_axis_1' and use it in 'y_ref'.
- MATH GENERATION: Include 'math' (top-level) and 'is_math' (series-level) if the scatter data was generated by a detectable mathematical function.
- HYBRID/THRESHOLDS: If the chart contains a horizontal or vertical threshold line, extract it as a separate series with exactly 2 data points representing its start and end.
- HYBRID/AREA BANDS: If the chart contains shaded area bands, error bars, or confidence intervals, represent them as a separate series where 'data' is a 3-element list: ["x_val", "y_max", "y_min"]. Do not calculate cluster statistics for area bands.
- Return ONLY the raw JSON object. Do not use markdown formatting blocks (```)."""


# ==========================================
# PHASE 1: INFERENCE & SAVING PREDICTIONS
# ==========================================
def run_inference_and_save():
    print(f"Loading Base Processor from {BASE_MODEL_ID}...")
    processor = AutoProcessor.from_pretrained(BASE_MODEL_ID)

    print("Loading Base Model into VRAM...")
    base_model = AutoModelForImageTextToText.from_pretrained(
        BASE_MODEL_ID,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        attn_implementation="sdpa"
    )

    print(f"Attaching LoRA Adapter from {ADAPTER_PATH}...")
    model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

    print("Merging weights for ultra-fast inference...")
    model = model.merge_and_unload()
    model.eval()
    print("✅ Model successfully assembled in memory!\n")

    predictions_record = []
    processed_ids = set()
    removed_count = 0

    if os.path.exists(PREDICTIONS_OUTPUT):
        try:
            with open(PREDICTIONS_OUTPUT, 'r', encoding="utf-8") as f:
                raw_records = json.load(f)
            
            for item in raw_records:
                if TARGET_CHART_TYPES and item.get("chart_type") in TARGET_CHART_TYPES:
                    removed_count += 1
                    continue
                
                predictions_record.append(item)
                processed_ids.add(item["id"])
                
            print(f"🔄 Resuming: Found {len(processed_ids)} valid existing predictions.")
            if removed_count > 0:
                print(f"🗑️ Removed {removed_count} predictions. They will be regenerated.")
                with open(PREDICTIONS_OUTPUT, 'w', encoding="utf-8") as f:
                    json.dump(predictions_record, f, indent=2, ensure_ascii=False)
                    
        except json.JSONDecodeError:
            print(f"⚠️ Warning: {PREDICTIONS_OUTPUT} was corrupted or empty. Starting from scratch.")

    with open(TEST_JSON_PATH, 'r', encoding="utf-8") as f:
        ground_truth_data = json.load(f)

    remaining_data = [item for item in ground_truth_data if item.get("id") not in processed_ids]
    print(f"📊 Total: {len(ground_truth_data)} | Already Done: {len(processed_ids)} | Remaining to Process: {len(remaining_data)}\n")

    for index, gt_item in enumerate(remaining_data, start=1):
        chart_id = gt_item.get("id")
        ctype = gt_item.get("chart_type")
        complexity = gt_item.get("complexity")
        img_path = os.path.join(TEST_IMG_DIR, f"{chart_id}.png")
        
        print(f"Generating [{len(processed_ids) + index}/{len(ground_truth_data)}] - {ctype} (ID: {chart_id})")
        
        if not os.path.exists(img_path):
            print(f"  ⚠️ Image not found: {img_path}. Skipping.")
            continue

        # 🎯 PRECISE SCHEMA ROUTING
        ctype_lower = ctype.lower()
        if "histogram" in ctype_lower:
            current_schema = SCHEMA_PROMPT_HISTOGRAM 
        elif "box" in ctype_lower:
            current_schema = SCHEMA_PROMPT_BOX 
        elif "scatter" in ctype_lower:
            current_schema = SCHEMA_PROMPT_SCATTER 
        elif "line" in ctype_lower:
            current_schema = SCHEMA_PROMPT_LINE
        elif "pie" in ctype_lower:
            current_schema = SCHEMA_PROMPT_PIE
        elif "area" in ctype_lower:
            current_schema = SCHEMA_PROMPT_AREA
        elif "bar" in ctype_lower:
            current_schema = SCHEMA_PROMPT_BAR
        
        messages = [
            {"role": "system", "content": "You are a chart data extraction specialist."},
            {"role": "user", "content": [
                {"type": "image", "image": img_path, "max_pixels": 768 * 768},
                {"type": "text", "text": current_schema} 
            ]}
        ]

        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        ).to("cuda")

        with torch.no_grad():
            generated_ids = model.generate(
                **inputs, 
                max_new_tokens=4096,
                temperature=0.1,  
                do_sample=False
            )

        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        prediction_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0]

        predictions_record.append({
            "id": chart_id,
            "chart_type": ctype,
            "complexity": complexity,
            "predicted_text": prediction_text
        })

        with open(PREDICTIONS_OUTPUT, 'w', encoding="utf-8") as f:
            json.dump(predictions_record, f, indent=2, ensure_ascii=False)

    print(f"\n✅ Selective generation complete! All predictions synchronized and saved to {PREDICTIONS_OUTPUT}")
    
    del model
    del base_model
    torch.cuda.empty_cache()

if __name__ == "__main__":
    run_inference_and_save()

Loading Base Processor from Qwen/Qwen2.5-VL-7B-Instruct...


The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
`torch_dtype` is deprecated! Use `dtype` instead!


Loading Base Model into VRAM...


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Attaching LoRA Adapter from /workspace/qwen_lora_final/qwen_lora_final...
Merging weights for ultra-fast inference...


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✅ Model successfully assembled in memory!

📊 Total: 258 | Already Done: 0 | Remaining to Process: 258

Generating [1/258] - Histogram (ID: 131740)
Generating [2/258] - Line Chart (ID: 149999)
Generating [3/258] - Line Chart (ID: 153147)
Generating [4/258] - Line Chart (ID: 149267)
Generating [5/258] - Line Chart (ID: 107744)
Generating [6/258] - Line Chart (ID: 90311)
Generating [7/258] - Area Chart (ID: 55210)
Generating [8/258] - Pie Chart (ID: 71634)
Generating [9/258] - Area Chart (ID: 122595)
Generating [10/258] - Scatter Plot (ID: 50003)
Generating [11/258] - Scatter Plot (ID: 118774)
Generating [12/258] - Box Plot (ID: 50419)
Generating [13/258] - Bar Chart (ID: 147731)
Generating [14/258] - Bar Chart (ID: 154422)
Generating [15/258] - Pie Chart (ID: 66936)
Generating [16/258] - Line Chart (ID: 56537)
Generating [17/258] - Scatter Plot (ID: 59868)
Generating [18/258] - Pie Chart (ID: 155348)
Generating [19/258] - Box Plot (ID: 105282)
Generating [20/258] - Line Chart (ID: 65440)

In [15]:
!pip install json-repair


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [27]:
TEST_JSON_PATH = "extracted_chart_specs_test.json"
TEST_IMG_DIR = "./test_images_300"
PREDICTIONS_OUTPUT = "predicted_specs_test.json"
METRICS_CSV = "stage1_evaluation_results_ours.csv"

# ==========================================
# PHASE 2: EVALUATION FROM SAVED FILES
# ==========================================
def run_evaluation():
    print(f"\nStarting Phase 2: Evaluating predictions from {PREDICTIONS_OUTPUT}...")
    
    with open(TEST_JSON_PATH, 'r', encoding="utf-8") as f:
        ground_truth_data = json.load(f)
        
    with open(PREDICTIONS_OUTPUT, 'r', encoding="utf-8") as f:
        predictions_data = json.load(f)

    # Convert ground truth to a dictionary for easy ID lookup
    gt_dict = {str(item["id"]): item for item in ground_truth_data if "id" in item}
    
    results = []

    for pred_item in predictions_data:
        chart_id = str(pred_item["id"])
        
        if chart_id not in gt_dict:
            continue
            
        gt_item = gt_dict[chart_id]
        
        # Extract the strings
        gt_spec_str = json.dumps(gt_item.get("ChartSpec", {}))
        pred_spec_str = pred_item["predicted_text"]

        #clean the json before evaluating
        clean_json_str = pred_spec_str.replace('\\"', '"').strip('"')
        
        # Run the rigorous evaluator
        metrics = evaluate_json_accuracy(gt_spec_str, clean_json_str)
        
        # Attach metadata for the CSV
        metrics["chart_id"] = chart_id
        metrics["chart_type"] = pred_item.get("chart_type", "Unknown")
        metrics["complexity"] = pred_item.get("complexity", "Unknown")
        
        results.append(metrics)

    # Save to CSV
    df = pd.DataFrame(results)
    
    # Ensure math_accuracy handles None values gracefully
    df['math_accuracy'] = df['math_accuracy'].astype(float) 
    
    df.to_csv(METRICS_CSV, index=False)

    print("\n" + "="*60)
    print("🎉 EVALUATION COMPLETE")
    print("="*60)

    # Define the exact metrics we want to aggregate
    metric_cols = [
        'valid_json', 'schema_score', 'topology_score', 
        'domain_accuracy', 'numerical_accuracy', 
        'categorical_accuracy', 'math_accuracy', 'semantic_score'
    ]

    # 1. Group by chart_type
    summary = df.groupby('chart_type')[metric_cols].mean().reset_index()

    # 2. Calculate OVERALL averages across the entire test set
    overall = df[metric_cols].mean().to_frame().T
    overall['chart_type'] = 'OVERALL'
    
    # Combine chart types with the overall row
    summary = pd.concat([summary, overall], ignore_index=True)

    # Format all metric columns as clean percentages for the terminal
    for col in metric_cols:
        summary[col] = (summary[col] * 100).apply(lambda x: f"{x:.1f}%" if pd.notnull(x) else "N/A")
    
    # Print the primary summary table
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    print("📊 Performance by Chart Type:")
    print("-" * 60)
    print(summary.to_string(index=False))
    
    # 3. Print a secondary breakdown by complexity
    print("\n📊 Performance by Complexity:")
    print("-" * 60)
    comp_summary = df.groupby('complexity')[['valid_json', 'numerical_accuracy', 'schema_score', 'semantic_score']].mean().reset_index()
    for col in ['valid_json', 'numerical_accuracy', 'schema_score', 'semantic_score']:
        comp_summary[col] = (comp_summary[col] * 100).apply(lambda x: f"{x:.1f}%" if pd.notnull(x) else "N/A")
    print(comp_summary.to_string(index=False))

    print(f"\n📁 Full detailed metrics saved to: {METRICS_CSV}")

if __name__ == "__main__":
    run_evaluation()


Starting Phase 2: Evaluating predictions from predicted_specs_test.json...

🎉 EVALUATION COMPLETE
📊 Performance by Chart Type:
------------------------------------------------------------
  chart_type valid_json schema_score topology_score domain_accuracy numerical_accuracy categorical_accuracy math_accuracy semantic_score
  Area Chart     100.0%        99.2%          73.8%           96.5%              77.0%                97.9%           N/A          77.6%
   Bar Chart     100.0%        99.3%          87.0%           92.6%              88.2%                94.6%           N/A          77.6%
    Box Plot     100.0%       100.0%          53.8%           77.3%              44.5%                84.7%           N/A          38.7%
   Histogram      77.8%        75.6%          61.1%           82.1%              35.5%                  N/A           N/A          28.6%
  Line Chart     100.0%        99.5%          95.5%           84.5%              85.4%                99.8%         76.1%     